# Pilcrow / ai-detector — launch retrain

**Two jobs in one run:**
1. **Unblock the sale.** Retrain **Stage-2 from `answerdotai/ModernBERT-large` (Apache-2.0)** instead of the
   unlicensed `Donnyed/LLM_Detector_Preview_model`. Fine-tuning a no-license model does not grant resale rights,
   so this is the one hard blocker before you can charge for Pilcrow.
2. **Stop flagging humans.** Train on the merged corpus that adds contemporary **conversation / q&a / casual**
   human registers (matched-pair, so no `casual = human` shortcut) while keeping the archaic + encyclopedic
   coverage the frozen OOD gate also tests.

Stage-1 (e5-small, **MIT** — already license-clean) is an **optional** second pass at the bottom; it is the
calibrated/abstaining stage and gives extra human-FP reduction. The blocker is Stage-2.

### Requirements
- **GPU runtime is mandatory.** Colab: *Runtime → Change runtime type → T4 GPU* (or better). The first cell
  hard-fails on a CPU runtime. ModernBERT-large LoRA on CPU is not acceptable.
- The converted Core ML model runs on the **Apple Neural Engine / GPU** (`cpu_and_ne`, FP16) — ModernBERT is
  ANE-eligible, unlike the old DeBERTa. Nothing here ships a CPU-only model.

### Files to upload (paths on your Mac, repo root `~/Code/pilcrow`)

| upload as | repo path | required |
|---|---|---|
| `train.csv` | `data/corpus/train.csv` | yes |
| `eval.csv` | `data/corpus/eval.csv` | yes |
| `ood_human.csv` | `data/corpus/ood_human.csv` | yes (the ship gate) |
| `crossgen_eval.csv` | `data/corpus/crossgen_eval.csv` | yes (recall probe) |
| `finetune-lora.py` | `scripts/finetune-lora.py` | yes |
| `audit-confound.py` | `scripts/audit-confound.py` | yes |
| `eval_held.csv` | `data/corpus/eval_held.csv` | optional |

## 0 · GPU check + upload

In [ ]:
import os, glob, shutil, torch

assert torch.cuda.is_available(), (
    'NO GPU. Runtime > Change runtime type > T4 GPU (or better), then Run all. '
    'A CPU runtime is not acceptable for this train.')
print('GPU:', torch.cuda.get_device_name(0))

REQUIRED = ['train.csv','eval.csv','ood_human.csv','crossgen_eval.csv',
            'finetune-lora.py','audit-confound.py']
OPTIONAL = ['eval_held.csv']

# Kaggle: pull from an attached dataset if present
for f in REQUIRED + OPTIONAL:
    if not os.path.exists(f):
        hits = glob.glob('/kaggle/input/**/' + f, recursive=True)
        if hits: shutil.copy(hits[0], f)

missing = [f for f in REQUIRED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Pick these', len(missing), 'files:', missing)
        files.upload()
    except Exception:
        print('Not on Colab and not found under /kaggle/input -> attach them.')

missing = [f for f in REQUIRED if not os.path.exists(f)]
assert not missing, 'STILL MISSING: ' + str(missing)
for f in REQUIRED + OPTIONAL:
    if os.path.exists(f): print('  ', f, os.path.getsize(f), 'bytes')
print('\nOK - all required files present')

## 1 · Dependencies
`transformers==4.49.0` matches the Mac-side Core ML convert. `torchao` 0.10 silently breaks peft LoRA on
Colab/Kaggle, so it is removed.

In [ ]:
!pip -q install 'transformers==4.49.0' 'peft>=0.11' 'accelerate>=0.30' scikit-learn datasets 2>/dev/null
!pip -q uninstall -y torchao 2>/dev/null
import importlib, torch
print('torchao removed:', importlib.util.find_spec('torchao') is None, '| CUDA:', torch.cuda.is_available())

## 1b · Mount Google Drive  (do this for a remote / long run)
A Colab session timeout wipes the VM's local disk. With Drive mounted, the download cells below ALSO copy the
zip into your Drive, so a disconnect can't lose the trained model. Skip only for a short run you'll babysit.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

## 2 · Gate helpers
`pai_table` = the **ship gate**: per-register false-positive rate on the frozen OOD humans at the shipped
decision threshold `P(AI) > 0.90`. `held_eval` (optional) = per-register FP + recall on the source-disjoint
held-out set, if you uploaded `eval_held.csv`.

In [ ]:
def pai_table(M, ood='ood_human.csv'):
    import torch, csv, numpy as np
    from collections import defaultdict
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    tok = AutoTokenizer.from_pretrained(M)
    model = AutoModelForSequenceClassification.from_pretrained(M).to(dev).eval()
    id2 = {int(k): v for k, v in model.config.id2label.items()}
    human_idx = next((i for i, n in id2.items() if 'human' in str(n).lower()), 0)
    rows = [r for r in csv.DictReader(open(ood)) if r.get('text','').strip()]
    pAI, regs = [], []
    with torch.no_grad():
        for i in range(0, len(rows), 32):
            ch = rows[i:i+32]
            enc = tok([r['text'] for r in ch], truncation=True, max_length=512, padding=True, return_tensors='pt').to(dev)
            p = torch.softmax(model(**enc).logits, -1).cpu()
            for pr, row in zip(p, ch):
                pAI.append(1.0 - pr[human_idx].item()); regs.append(row.get('register','?'))
    pAI = np.array(pAI)
    print(f'[{M}] OOD humans={len(pAI)}  mean P(AI)={pAI.mean():.3f}')
    for tau in [0.5,0.7,0.85,0.9,0.93,0.95]:
        print(f'  P(AI)>{tau:.2f}: {int((pAI>tau).sum()):3d}/{len(pAI)} = {(pAI>tau).mean()*100:4.1f}%')
    agg = defaultdict(lambda:[0,0])
    for p,reg in zip(pAI,regs): agg[reg][1]+=1; agg[reg][0]+=int(p>0.90)
    print('  per-register FP at P(AI)>0.90  (this is the ship gate):')
    for reg in sorted(agg, key=lambda k:-agg[k][0]/max(1,agg[k][1])):
        f,t = agg[reg]; print(f'    {reg[:18]:18s} {f}/{t} = {f/t*100:4.0f}%')

In [ ]:
def held_eval(M, path='eval_held.csv'):
    import os, torch, csv
    from collections import defaultdict
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    if not os.path.exists(path):
        print('eval_held.csv not uploaded - skipping (optional).'); return
    dev='cuda' if torch.cuda.is_available() else 'cpu'
    tok=AutoTokenizer.from_pretrained(M)
    model=AutoModelForSequenceClassification.from_pretrained(M).to(dev).eval()
    id2={int(k):v for k,v in model.config.id2label.items()}
    hidx=next((i for i,n in id2.items() if 'human' in str(n).lower()),0)
    rows=[r for r in csv.DictReader(open(path)) if r.get('text','').strip()]
    P=[]
    with torch.no_grad():
        for i in range(0,len(rows),32):
            ch=rows[i:i+32]
            enc=tok([r['text'] for r in ch],truncation=True,max_length=512,padding=True,return_tensors='pt').to(dev)
            pr=torch.softmax(model(**enc).logits,-1).cpu()
            for p in pr: P.append(1.0-p[hidx].item())
    agg=defaultdict(lambda:[0,0,0,0])
    for r,p in zip(rows,P):
        a=agg[r['register']]
        if str(r['label'])=='0': a[1]+=1; a[0]+=int(p>0.90)
        else: a[3]+=1; a[2]+=int(p>0.90)
    print(f'[{M}] held-out @ P(AI)>0.90 - FP=human flagged, REC=AI caught')
    for reg in sorted(agg):
        hfp,hn,rec,an=agg[reg]
        print(f'  {reg:15s} FP {hfp:3d}/{hn:<3d} = {hfp/max(1,hn)*100:4.0f}%    REC {rec:3d}/{an:<3d} = {rec/max(1,an)*100:4.0f}%')

# Stage-2 — license-clean ModernBERT-large  ⟵ the blocker

Base = **`answerdotai/ModernBERT-large`** (Apache-2.0). It ships as a masked-LM with **no detection head**, so
`transformers` adds a fresh 2-class head — you **will** see a warning like *"Some weights of
ModernBertForSequenceClassification were not initialized: classifier.weight ..."*. That is expected; the head is
trained here. Labels are `0=human, 1=AI`, and the Mac converter exports `P(AI)=softmax[1]` (`ai_label_index=1`).

### 2a · Fine-tune  (2 epochs)
**Runtime: ~3 h on a free T4** (ModernBERT-large, ~4,400 steps, no Flash Attention). A faster GPU is the real
lever — ≈1 h on an L4, ≈20-30 min on an A100 (Colab Pro, or a rented A100 at ~$1/hr ≈ $0.40 a run). For a paid
launch this is worth not fighting the T4.

**2 epochs** is the stop: in the first run epoch 2 was the balanced sweet spot (in-dist FPR ~1.8%, recall
~94.5%) and epoch 3 over-fit (per-step loss → 0, recall slipping). The cell below persists the model to Drive
the instant it finishes, so a post-training timeout can't cost you the 3 hours again.

In [ ]:
!python finetune-lora.py \
  --base answerdotai/ModernBERT-large \
  --train train.csv --eval eval.csv \
  --seq 512 --batch 8 --epochs 2 --lr 2e-4 --save-steps 300 \
  --out out/stage2-ft

In [ ]:
# persist the finished model to Drive IMMEDIATELY (before gating) so a timeout can't eat the run
import os, shutil
if os.path.isdir('/content/drive/MyDrive') and os.path.isdir('out/stage2-ft/merged'):
    dst = '/content/drive/MyDrive/pilcrow-stage2-ft-merged'
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree('out/stage2-ft/merged', dst)
    print('SAFE: Stage-2 model copied to', dst)
else:
    print('Drive NOT mounted — model only on the VM disk. Mount Drive (cell 1b) before a long run!')

### 2a-recover · Only if the run was interrupted (GPU quota) — merge on CPU
`--save-steps` overwrites a single `out/stage2-ft/ckpt` dir, so this rebuilds `merged` from the **latest step
reached**. Pure weight math, no GPU. Skip this cell if 2a finished normally.

In [ ]:
# run ONLY if finetune-lora.py above did not reach the end (no out/stage2-ft/merged)
import torch
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
_b = AutoModelForSequenceClassification.from_pretrained('answerdotai/ModernBERT-large', num_labels=2, torch_dtype=torch.float32)
_m = PeftModel.from_pretrained(_b, 'out/stage2-ft/ckpt').merge_and_unload()
_m.save_pretrained('out/stage2-ft/merged')
try:    AutoTokenizer.from_pretrained('out/stage2-ft/ckpt').save_pretrained('out/stage2-ft/merged')
except Exception: AutoTokenizer.from_pretrained('answerdotai/ModernBERT-large').save_pretrained('out/stage2-ft/merged')
print('recovered -> out/stage2-ft/merged')

### 2b · OOD ship gate — the merged model on the frozen humans
Reference bars from prior runs: raw Donnyed base ≈ **35%** OOD FP; the last shipped (license-tainted) fine-tune
≈ **3%**. The license-clean model should land in that low range. **Ship only if** the per-register FP at
`P(AI)>0.90` is low across the board (especially `conversation`, `news`, `qa`, `encyclopedic`) **and** no single
register blows up. (The raw unlicensed base is intentionally not loaded here — keep the run license-clean.)

In [ ]:
pai_table('out/stage2-ft/merged')

### 2c · Cross-generator recall (held-out GPT / Llama / Mixtral — never trained on)
Confirms the model still catches AI from generators it never saw, i.e. it did not overcorrect to 'all human'.

In [ ]:
!python audit-confound.py --crossgen crossgen_eval.csv --model out/stage2-ft/merged

### 2d · Held-out per-register generalization (optional)

In [ ]:
held_eval('out/stage2-ft/merged')

### 2e · Download the Stage-2 model

In [ ]:
import os, shutil
shutil.make_archive('stage2-ft', 'zip', 'out/stage2-ft/merged')
print('zipped:', os.path.getsize('stage2-ft.zip'), 'bytes')
if os.path.isdir('/content/drive/MyDrive'):
    shutil.copy('stage2-ft.zip', '/content/drive/MyDrive/'); print('saved to Drive/MyDrive (survives timeouts)')
try:
    from google.colab import files; files.download('stage2-ft.zip')
except Exception:
    print('Kaggle: find stage2-ft.zip in the working dir / Output tab.')

# Stage-1 — e5-small  (optional, recommended for the human-FP fix)

Base = `MayZhou/e5-small-lora-ai-generated-detector` (**MIT** — already license-clean, not a blocker). 33M params,
trains in a few minutes. This is the calibrated/abstaining stage, so it does the most to pull contemporary-human
prose below the flag threshold. Run it if you also want the FP fix on the fast path, not just Stage-2.

### 1a · Fine-tune

In [ ]:
!python finetune-lora.py \
  --train train.csv --eval eval.csv \
  --seq 512 --batch 32 --epochs 3 --lr 2e-4 --save-steps 300 \
  --out out/stage1-ft

In [ ]:
import os, shutil
if os.path.isdir('/content/drive/MyDrive') and os.path.isdir('out/stage1-ft/merged'):
    dst = '/content/drive/MyDrive/pilcrow-stage1-ft-merged'
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree('out/stage1-ft/merged', dst)
    print('SAFE: Stage-1 model copied to', dst)

### 1b · OOD ship gate

In [ ]:
pai_table('out/stage1-ft/merged')

### 1c · Cross-generator recall

In [ ]:
!python audit-confound.py --crossgen crossgen_eval.csv --model out/stage1-ft/merged

### 1d · Held-out per-register (optional)

In [ ]:
held_eval('out/stage1-ft/merged')

### 1e · Download the Stage-1 model

In [ ]:
import os, shutil
shutil.make_archive('stage1-ft', 'zip', 'out/stage1-ft/merged')
print('zipped:', os.path.getsize('stage1-ft.zip'), 'bytes')
if os.path.isdir('/content/drive/MyDrive'):
    shutil.copy('stage1-ft.zip', '/content/drive/MyDrive/'); print('saved to Drive/MyDrive (survives timeouts)')
try:
    from google.colab import files; files.download('stage1-ft.zip')
except Exception:
    print('Kaggle: find stage1-ft.zip in the working dir / Output tab.')

# On your Mac — convert (ANE/NPU), calibrate, install

```bash
cd ~/Code/pilcrow            # repo root (folder is 'pilcrow', remote is ai-detector)
pip install 'transformers==4.49.0' 'coremltools>=9'

# ---- Stage-2 (the blocker) ----
mkdir -p out/stage2-ft/merged && unzip -o ~/Downloads/stage2-ft.zip -d out/stage2-ft/merged
python3 scripts/convert-stage2-modernbert.py out/stage2-ft/merged
#   -> writes Models/Stage2/ . Verify the printout says compute_units = cpu_and_ne
#      (ANE/GPU). If it fell back to cpu_only, FP16 parity NaN'd - tell Claude.

# License truth: this model now derives ONLY from Apache-2.0 weights.
python3 - <<'PY'
import json, pathlib
p = pathlib.Path('Models/Stage2/model-info.json'); d = json.loads(p.read_text())
d['license'] = 'apache-2.0'
d['base_model'] = 'answerdotai/ModernBERT-large'
d['source_repo'] = 'answerdotai/ModernBERT-large'
p.write_text(json.dumps(d, indent=2))
print('Stage-2 model-info: license', d['license'], '| ai_label_index', d.get('ai_label_index'),
      '| compute_units', d.get('compute_units'))
PY

# ---- Stage-1 (only if you trained it) ----
mkdir -p out/stage1-ft/merged && unzip -o ~/Downloads/stage1-ft.zip -d out/stage1-ft/merged
python3 scripts/convert-model.py out/stage1-ft/merged          # writes into Models/
python3 scripts/calibrate.py --model out/stage1-ft/merged --data data/corpus/calib.csv
#   -> paste the printed calibration block into Models/model-info.json by hand
#      (convert overwrites that file and drops the calibration).

# ---- deploy + relaunch the installed app ----
bash scripts/install.sh
```

Then re-run the bundled separation smoke check / open the app and paste a casual human message to confirm it no
longer over-flags. If the OOD gate above did **not** clear, the disciplined move is to ship on the abstention
wedge (flag only the most decisive cases, say *unknown* loudly) rather than start another retrain round.